# Environmental conditions and dengue forecasting — Iquitos, Peru
### Complete research notebook · Step by step

**Topic 3: Health Impact and Epidemic Emergence**  
**Research question:** *Do climate and vegetation measurements improve one-week-ahead predictions of dengue cases in Iquitos, Peru, compared with using recent cases and seasonality alone?*

This continues the organization and teaching style of your original **Iquitos DengAI Research Starter** notebook, but takes the work **from data understanding all the way through internal model selection and external-period evaluation**.

**What you will learn:** each variable and dataset; why we check missing values; how EDA relates to environmental epidemiology; what one-week-ahead forecasting means; how models are compared fairly; and why the later CDC observations are evaluated **only after** choosing a model.

**Important:** We predict **weekly reported case counts**, not population-standardized incidence rates. These statistical relationships are **not causal evidence** that rainfall or vegetation caused a dengue outbreak.

**Robustness edition:** adds expanding-window stability, an approximate paired block-bootstrap uncertainty analysis, and extreme-week sensitivity. No outbreak-classification section.

## Repository edition

Organized from the saved 15 November 2026 research package. Only paths, dataset exports, and setup documentation were adapted; the model specifications and evaluation procedure are preserved. Stored tables and figures are archived outputs from that package.


## Research roadmap

| Part | What we do | Research purpose |
|---|---|---|
| 0 | Setup and define data locations | Make the notebook reproducible |
| 1 | Load and merge DengAI training files | Get predictors and observed cases on the same row |
| 2 | Understand and clean the variables | Prepare transparent measurements |
| 3 | Exploratory Data Analysis (EDA) | Describe outbreaks, weather, and vegetation |
| 4 | Define the one-week-ahead forecast | Prevent future information entering predictions |
| 5 | Create past-week predictors | Build features known at prediction time |
| 6 | Internal validation (2008–2010) | Compare models and **choose one** |
| 7 | Freeze the choice; check CDC observations | Keep later observed counts out of selection |
| 8 | External-period evaluation (2010–2013) | Evaluate chosen model on later weeks |
| 9 | Interpret and save research outputs | Prepare figures and results for the report |

**Research rule:** We decide the candidate model specifications and evaluate them on the **same dates**. We do not pick a new model after looking at external results.



## 0 · Setup and required files

The four source CSV files below are stored in `data/raw/`. Run this notebook from the repository root or `notebooks/`. Tables are written to `reports/tables/`, figures to `reports/figures/`, and prepared datasets to `data/processed/`.

- `dengue_features_train.csv` — environmental and time measurements in the labeled period.
- `dengue_labels_train.csv` — true weekly dengue counts in the labeled period.
- `dengue_features_test.csv` — weather/vegetation observations in later weeks (no case counts).
- `Iquitos_CDC_Observed_Cases_2000_2013.csv` — published surveillance case counts for verifying and evaluating the later period.

**Sources:** DrivenData DengAI (features and labels); CDC Epidemic Prediction Initiative (Iquitos case counts), https://cdcepi.github.io/dengue/data.html. The CDC file in `data/raw/` contains a 2-column extract (`week_start_date`, `total_cases_cdc`) of the public case series.

**Python packages:** `pandas`, `numpy`, `matplotlib`, `scikit-learn`, `IPython`. Install packages only if your kernel is missing them: `pip install pandas numpy matplotlib scikit-learn ipython`.

The eight-model comparison also uses `xgboost` (`%pip install xgboost` if your environment lacks the package).


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, PoissonRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, precision_score, recall_score, f1_score
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.figsize": (10, 4.3), "figure.dpi": 105})

# Locate the repository whether Jupyter starts at the root or in notebooks/.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data" / "raw" / "dengue_features_train.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the repository.")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

files_needed = [
    "dengue_features_train.csv", "dengue_labels_train.csv",
    "dengue_features_test.csv", "Iquitos_CDC_Observed_Cases_2000_2013.csv"
]
missing_files = [x for x in files_needed if not (DATA_DIR / x).exists()]
if missing_files:
    raise FileNotFoundError(
        f"Missing files: {missing_files}. Put all four CSV files in data/raw/."
    )

OUTPUT_DIR = PROJECT_ROOT / "reports" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Reading CSV files from:", DATA_DIR.resolve())
print("Results will be saved to:", OUTPUT_DIR.resolve())

## 1 · Load and understand our original data

For supervised prediction we need **features (X)** and **labels (Y)**.

- **Features:** rainfall, temperature, humidity, vegetation, dates, etc.
- **Label:** `total_cases`, the number of reported dengue cases in a week.

There are two cities in DengAI, but **we study only Iquitos (`iq`)**. Do not mix city-year-week rows accidentally.

In [ ]:
features_train = pd.read_csv(DATA_DIR / "dengue_features_train.csv")
labels_train = pd.read_csv(DATA_DIR / "dengue_labels_train.csv")
features_test = pd.read_csv(DATA_DIR / "dengue_features_test.csv")

print("Original features (training):", features_train.shape)
print("Original labels (training):  ", labels_train.shape)
print("Later features (test):      ", features_test.shape)
print("Training observations by city:")
display(features_train.groupby("city").size().rename("weeks").to_frame())

### 1.1 · Merge the features with observed dengue cases

One row corresponds to **one week in one city**. The combination `(city, year, weekofyear)` must be unique. Merging joins the weather variables and `total_cases` on the **same weekly observation**.

`assert` statements are data-integrity checks: they stop the notebook if a key is duplicated or any week is lost in the merge.

In [ ]:
keys = ["city", "year", "weekofyear"]
assert not features_train.duplicated(keys).any(), "Duplicate feature keys"
assert not labels_train.duplicated(keys).any(), "Duplicate label keys"

combined = features_train.merge(
    labels_train, on=keys, how="inner", validate="one_to_one"
)
assert len(combined) == len(features_train) == len(labels_train), "Merge lost rows"

iq = combined.loc[combined["city"].eq("iq")].copy()
iq["week_start_date"] = pd.to_datetime(iq["week_start_date"])
iq = iq.sort_values("week_start_date").reset_index(drop=True)
assert len(iq) == 520, "Unexpected number of labeled Iquitos weeks"
assert iq["week_start_date"].is_unique

print("Iquitos observations:", len(iq))
print("Date range:", iq.week_start_date.min().date(), "to", iq.week_start_date.max().date())
print("Mean weekly cases:", round(iq.total_cases.mean(), 2))
print("Weeks with 0 cases:", int(iq.total_cases.eq(0).sum()))
display(iq[["week_start_date", "total_cases", "precipitation_amt_mm", "reanalysis_avg_temp_k"]].head())

## 2 · Understand the environmental variables

| Column in CSV | What it measures | Unit | Why we might care |
|---|---|---|---|
| `total_cases` | Reported dengue cases in that week | Count | **Target** we want to predict |
| `precipitation_amt_mm` | Satellite-derived weekly precipitation | mm | Potential mosquito breeding conditions |
| `reanalysis_avg_temp_k` | Average air temperature in reanalysis | Kelvin | Mosquito and viral biology |
| `reanalysis_relative_humidity_percent` | Relative humidity | % | Mosquito survival and activity |
| `ndvi_ne`, `ndvi_nw`, `ndvi_se`, `ndvi_sw` | Vegetation greenness in four satellite cells | NDVI index | Environmental and habitat characteristics |
| `weekofyear` | Calendar week | 1–53 | Helps represent annual seasonality |

**Transformations we'll make:**
- `temperature_c = reanalysis_avg_temp_k − 273.15` for Celsius.
- `ndvi_mean` = average of the four NDVI cells, allowing missing pixels when possible.

**NDVI is not a direct measure of environmental degradation or deforestation**. It indicates greenness. Likewise, these climate variables cannot prove a causal effect on dengue cases.

In [ ]:
iq["rain_mm"] = iq["precipitation_amt_mm"]
iq["temperature_c"] = iq["reanalysis_avg_temp_k"] - 273.15
iq["humidity_pct"] = iq["reanalysis_relative_humidity_percent"]
ndvi_columns = ["ndvi_ne", "ndvi_nw", "ndvi_se", "ndvi_sw"]
iq["ndvi_mean"] = iq[ndvi_columns].mean(axis=1)

columns_for_missing = [
    "total_cases", "rain_mm", "temperature_c", "humidity_pct", "ndvi_mean"
]
missing_summary = pd.DataFrame({
    "missing_n": iq[columns_for_missing].isna().sum(),
    "missing_pct": (100 * iq[columns_for_missing].isna().mean()).round(2)
})
print("Missing values (Iquitos, 520 weeks):")
display(missing_summary)
print("\nBasic descriptive statistics:")
display(iq[columns_for_missing].describe().round(2))

### 2.1 · What do missing values mean?

- `missing_n` = how many weeks have an unavailable measurement.
- `missing_pct` = percentage of those weeks (`missing_n / 520 × 100`).
- A missing rainfall measurement **does not mean 0 mm of rain**.

We don't replace missing values using the full time series at this stage. During modelling, the **median for each predictor is learned using fitting rows only**, to avoid looking ahead into validation or external periods.

## 3 · Exploratory Data Analysis (EDA)

EDA is the **descriptive** stage: what happened to dengue cases, rainfall, humidity and vegetation over time? The graphs do **not** by themselves explain why outbreaks occurred.

### 3.1 · How did weekly dengue cases change over time?

In [ ]:
fig, ax = plt.subplots()
ax.plot(iq["week_start_date"], iq["total_cases"], linewidth=1)
ax.set(title="Iquitos — weekly reported dengue cases", xlabel="Year", ylabel="Cases per week")
plt.tight_layout()
plt.show()
print("Largest reported weekly count:")
display(iq.nlargest(3, "total_cases")[["week_start_date", "total_cases"]])

**How to interpret:** Look for *when* and *how large* the peaks were. A very high reported count may reflect a true outbreak but can also be influenced by diagnostic and reporting practices. The graph alone does not tell us what caused any specific peak.

### 3.2 · Is dengue seasonal?

In [ ]:
seasonal = iq.groupby("weekofyear", as_index=False)["total_cases"].mean()
fig, ax = plt.subplots()
ax.plot(seasonal["weekofyear"], seasonal["total_cases"], marker=".", linewidth=1)
ax.set(title="Average dengue cases by week of year", xlabel="Week of year", ylabel="Mean cases (across years)")
plt.tight_layout()
plt.show()

**How to interpret:** The same calendar week is averaged across all observed years. Peaks around particular weeks may indicate seasonality, but they can also be inflated by a single large epidemic. **Seasonality is useful to predict**, even before we include weather.

### 3.3 · Weekly precipitation

In [ ]:
fig, ax = plt.subplots()
ax.plot(iq["week_start_date"], iq["rain_mm"], linewidth=1)
ax.set(title="Iquitos — weekly precipitation estimate", xlabel="Year", ylabel="Precipitation (mm)")
plt.tight_layout()
plt.show()

### 3.4 · Weekly temperature

In [ ]:
fig, ax = plt.subplots()
ax.plot(iq["week_start_date"], iq["temperature_c"], linewidth=1)
ax.set(title="Iquitos — weekly mean reanalysis temperature", xlabel="Year", ylabel="Temperature (°C)")
plt.tight_layout()
plt.show()

### 3.5 · Weekly relative humidity

In [ ]:
fig, ax = plt.subplots()
ax.plot(iq["week_start_date"], iq["humidity_pct"], linewidth=1)
ax.set(title="Iquitos — weekly reanalysis relative humidity", xlabel="Year", ylabel="Relative humidity (%)")
plt.tight_layout()
plt.show()

### 3.6 · Average satellite vegetation index (NDVI)

In [ ]:
fig, ax = plt.subplots()
ax.plot(iq["week_start_date"], iq["ndvi_mean"], linewidth=1)
ax.set(title="Iquitos — mean NDVI across four satellite cells", xlabel="Year", ylabel="Mean NDVI")
plt.tight_layout()
plt.show()

### 3.7 · EDA notes for the research report

Record *observations*, not assumed causes. For example:

> Weekly dengue cases vary considerably through time, with several pronounced outbreak peaks. Higher mean case counts in some weeks of the year suggest a seasonal component. Rainfall, temperature, humidity and NDVI also fluctuate, but the graphs cannot establish whether they caused the outbreaks. We therefore test whether **past environmental measurements add predictive information**.

**Important:** To avoid choosing models based on the later evaluation period, we will specify a **small fixed candidate set** below rather than exhaustively searching through many lags or methods.

## 4 · Define the forecasting task **before** fitting models

### The target is one-week-ahead **reported cases**

For a week labeled **t**, we want to predict `total_cases` in that week using information available by the end of **t−1**:

| Target to predict | Predictors available before it |
|---|---|
| Dengue cases in **week t** | Dengue counts from **t−1 and t−2** |
|  | Rainfall in **t−1 and t−2** |
|  | Temperature, humidity, NDVI from **t−1** |
|  | Calendar week of **t** (known in advance) |

**One-week horizon:** We repeatedly predict the next week, then, after that week passes, the observed case count can be used for the following forecast. We do **not** use `total_cases` from the week being predicted.

### Internal vs external evaluation

- **Internal:** first 416 of the 520 labeled weeks are the fitting period; last **104 weeks** are for model selection.
- **External:** 156 additional weeks (2010–2013) will be examined **only after** we have locked a selected model.
- No random train/test split. Training always precedes evaluation chronologically.

**Important limitation:** This is a *retrospective rolling one-week-ahead* protocol. It assumes newly observed dengue counts become available promptly. It is **not** a 156-week forecast issued all at once in June 2010.

## 5 · Feature engineering: information from previous weeks

*Lag* means a value from an earlier period. For example, `cases_lag1` is last week's dengue count and `rain_lag2` is rainfall two weeks ago.

We use **log(1 + cases)** for the past-case predictors because dengue counts can have very large outbreak peaks.

We represent seasonal timing with **sine and cosine**, so the end and beginning of the year are treated as nearby points in the annual cycle.

In [ ]:
def make_features(frame):
    d = frame.sort_values("week_start_date").reset_index(drop=True).copy()
    d["temperature_c"] = d["reanalysis_avg_temp_k"] - 273.15
    d["ndvi_mean"] = d[ndvi_columns].mean(axis=1)

    d["cases_lag1"] = d["total_cases"].shift(1)
    d["cases_lag2"] = d["total_cases"].shift(2)
    d["log_cases_lag1"] = np.log1p(d["cases_lag1"])
    d["log_cases_lag2"] = np.log1p(d["cases_lag2"])

    d["rain_lag1"] = d["precipitation_amt_mm"].shift(1)
    d["rain_lag2"] = d["precipitation_amt_mm"].shift(2)
    d["temp_lag1"] = d["temperature_c"].shift(1)
    d["humidity_lag1"] = d["reanalysis_relative_humidity_percent"].shift(1)
    d["ndvi_lag1"] = d["ndvi_mean"].shift(1)

    angle = 2 * np.pi * (d["weekofyear"] - 1) / 52.1775
    d["season_sin"] = np.sin(angle)
    d["season_cos"] = np.cos(angle)
    return d

history_features = ["log_cases_lag1", "log_cases_lag2", "season_sin", "season_cos"]
climate_features = history_features + ["rain_lag1", "rain_lag2", "temp_lag1", "humidity_lag1"]
vegetation_features = climate_features + ["ndvi_lag1"]

model_df = make_features(iq).dropna(subset=["cases_lag1", "cases_lag2"]).copy()
print("Weeks usable after the 2-week lag warm-up:", len(model_df))
cols = ["week_start_date", "total_cases", "cases_lag1", "rain_lag1", "temp_lag1", "ndvi_lag1"]
display(model_df[cols].head(6).round(2))
# Save reusable historical analysis tables.
iq.to_csv(PROCESSED_DIR / "iquitos_history.csv", index=False)
model_df.to_csv(PROCESSED_DIR / "iquitos_model_features.csv", index=False)


### 5.1 · Check the timing: no target leakage

For each row, **`total_cases` is the answer**, while `cases_lag1` and environmental lags refer to earlier weeks. It is fine to know the *date* of the target week in advance.

The first two rows cannot have complete case lags, so we remove them. Missing weather values later on will be imputed by a training-only preprocessing pipeline.

In [ ]:
for col in ["cases_lag1", "cases_lag2"]:
    assert model_df[col].notna().all()
assert (model_df["week_start_date"].diff().dropna().dt.days.between(6, 9)).all(), "Check weekly spacing"
assert len(model_df) == 518
print("Feature timing and weekly order checks: passed")
print("Note: a few year-boundary dates differ by 8–9 days due to source date conventions.")

## 6 · Internal validation — compare models fairly

### 6.1 Chronological split

The original **first 416 weeks** are the fitting period (the first two are lost to lag warm-up), and the **last 104 weeks** are validation. All candidate models use exactly the same split.

### What we compare

| Candidate | Information used | What it tests |
|---|---|---|
| Persistence | Last week's dengue cases | Very simple benchmark |
| Ridge: history + season | Past dengue + seasonal sine/cosine | What case history predicts without weather |
| Ridge: history + climate | Above + past rain, temperature, humidity | **Main environmental hypothesis** |
| Ridge: history + climate + NDVI | Above + vegetation index | Whether vegetation adds value |
| Poisson: history + climate | A model designed for nonnegative counts | Alternative regression family |

`Ridge` is regularized linear regression on `log(1+cases)`; `Poisson` models count outcomes with a log link. Predicted case counts are **clipped to be nonnegative** before scoring. We fix the model hyperparameters in advance, rather than repeatedly tuning them on the same holdout.

**Selection metric:** lowest **MAE** (Mean Absolute Error). We also report RMSE, which penalizes large missed outbreaks more strongly.

In [ ]:
cutoff = iq.iloc[416]["week_start_date"]
fit_df = model_df.loc[model_df["week_start_date"] < cutoff].copy()
val_df = model_df.loc[model_df["week_start_date"] >= cutoff].copy()
assert len(fit_df) == 414 and len(val_df) == 104
print("Fit period:", fit_df.week_start_date.min().date(), "→", fit_df.week_start_date.max().date(), "|", len(fit_df), "rows")
print("Validation:", val_df.week_start_date.min().date(), "→", val_df.week_start_date.max().date(), "|", len(val_df), "rows")

### 6.2 Build preprocessing + regression pipelines

A **pipeline** ensures the same preparation is used consistently:

1. Replace missing predictors by the **median learned in the fitting period**.
2. Standardize predictors using the **fitting-period mean and standard deviation**.
3. Fit the regression on those transformed inputs.

This avoids leakage from validation-period information into preprocessing.

We compare **eight candidates**, including shallow XGBoost models. All choices use only the internal period for selection. The seasonal sine/cosine predictors are included in all Ridge and XGBoost feature sets, including those named `history + climate`.

In [ ]:
def ridge_log_model():
    return TransformedTargetRegressor(
        regressor=Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=12.0))
        ]),
        func=np.log1p, inverse_func=np.expm1
    )

def poisson_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("poisson", PoissonRegressor(alpha=1.0, max_iter=1200))
    ])

def xgb_model():
    # Shallow trees limit overfitting in this relatively small weekly dataset.
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("xgb", XGBRegressor(
            objective="count:poisson", n_estimators=150, max_depth=2,
            learning_rate=0.03, min_child_weight=5,
            subsample=0.8, colsample_bytree=0.8,
            reg_lambda=5.0, tree_method="hist", random_state=42, n_jobs=1
        ))
    ])

candidate_configs = {
    "Persistence: previous week": {"kind": "naive", "features": []},
    "Ridge: history + season": {"kind": "ridge", "features": history_features},
    "Ridge: history + climate": {"kind": "ridge", "features": climate_features},
    "Ridge: history + climate + NDVI": {"kind": "ridge", "features": vegetation_features},
    "Poisson: history + climate": {"kind": "poisson", "features": climate_features},
    "XGBoost: history + season": {"kind": "xgboost", "features": history_features},
    "XGBoost: history + climate": {"kind": "xgboost", "features": climate_features},
    "XGBoost: history + climate + NDVI": {"kind": "xgboost", "features": vegetation_features},
}

def fit_candidate(spec, data):
    if spec["kind"] == "naive":
        return None
    if spec["kind"] == "ridge":
        estimator = ridge_log_model()
    elif spec["kind"] == "poisson":
        estimator = poisson_model()
    elif spec["kind"] == "xgboost":
        estimator = xgb_model()
    else:
        raise ValueError(f"Unknown model type: {spec['kind']}")
    estimator.fit(data[spec["features"]], data["total_cases"])
    return estimator

def predict_candidate(estimator, spec, data):
    if spec["kind"] == "naive":
        yhat = data["cases_lag1"].to_numpy(dtype=float)
    else:
        yhat = estimator.predict(data[spec["features"]])
    return np.clip(np.asarray(yhat, dtype=float), 0, None)

def score_forecast(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
    }

print("Number of prespecified candidate models:", len(candidate_configs))

### 6.3 Fit each model and compare on the same 104 later weeks

The model sees outcomes from the fitting period when its coefficients are estimated. During the **rolling** validation simulation, the prior week's **actual** case count is permitted as an input only once that week has elapsed. The model is **not refitted** during these 104 weeks.

The models are scored against the actual cases in those 104 target weeks.

In [ ]:
internal_models = {}
internal_predictions = {}
internal_rows = []

for name, spec in candidate_configs.items():
    fitted = fit_candidate(spec, fit_df)
    forecast = predict_candidate(fitted, spec, val_df)
    internal_models[name] = fitted
    internal_predictions[name] = forecast
    internal_rows.append({"Model": name, **score_forecast(val_df["total_cases"], forecast)})

internal_results = pd.DataFrame(internal_rows).sort_values(["MAE", "RMSE"]).reset_index(drop=True)
selected_model_name = internal_results.loc[0, "Model"]
selected_spec = candidate_configs[selected_model_name]

print("Selected using INTERNAL MAE only:", selected_model_name)
display(internal_results.round(3))
internal_results.to_csv(OUTPUT_DIR / "internal_model_comparison.csv", index=False)

### 6.4 Visual check: internal predictions versus observed cases

A low average error does not guarantee that the model predicts **outbreak peaks** well. Compare the line charts and look for large forecast errors when cases increase suddenly.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.4))
ax.plot(val_df["week_start_date"], val_df["total_cases"], label="Observed cases", linewidth=2)
ax.plot(val_df["week_start_date"], internal_predictions["Persistence: previous week"], label="Previous-week benchmark", alpha=0.70)
ax.plot(val_df["week_start_date"], internal_predictions[selected_model_name], label=f"Selected model: {selected_model_name}", alpha=0.90)
ax.set(title="Internal validation (104 weeks) — observed vs predicted", xlabel="Date", ylabel="Weekly dengue cases")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "internal_validation.png", dpi=170)
plt.show()

### 6.5 Write down the result **before** checking the CDC period

Answer these questions in your report:

- **Which model had the smallest internal MAE?** (Use the table above.)
- **How much lower was its MAE than the history-only model?** Is the difference large or small?
- **Did NDVI improve the score?** Do not assume that vegetation helps just because it is ecologically relevant.
- **Were large epidemic weeks underestimated?** MAE may obscure errors at rare peaks.

We now **freeze the selected model specification**. The external-period data must **not** be used to revise this choice.

In [ ]:
internal_history_mae = internal_results.loc[
    internal_results["Model"] == "Ridge: history + season", "MAE"
].iloc[0]
internal_winner_mae = internal_results.loc[
    internal_results["Model"] == selected_model_name, "MAE"
].iloc[0]
print("LOCKED MODEL CHOICE:", selected_model_name)
print(f"Internal MAE improvement over history-only Ridge: {internal_history_mae - internal_winner_mae:+.3f} cases/week")
# No further model choices below this cell.

### 6.6 · Robustness check 1 — Expanding-window validation

Our original comparison was based on **one** 104-week validation period. That is not enough to establish that including environmental predictors consistently improves forecasts.

We now repeat the same *fixed* regression specifications across three **non-overlapping** 104-week validation blocks. Training always uses only earlier observations, and the fitting sample expands each time:

| Validation block | Historical training used | Weeks evaluated |
|---|---|---|
| Block A | First 208 weeks (minus 2-week lag warm-up) | Next 104 weeks, around 2004–2006 |
| Block B | First 312 weeks (minus warm-up) | Next 104 weeks, around 2006–2008 |
| Block C | First 416 weeks (minus warm-up) | Last 104 original labeled weeks, around 2008–2010 |

**What is held fixed?** All feature definitions, Ridge regularization (`alpha=12`), Poisson settings, preprocessing and the one-week forecast horizon. We do **not** use the 2010–2013 CDC period for these calculations. The exercise tests stability, not new model selection.

In [ ]:
# The index of model_df corresponds to the original Iquitos week index (2..519).
# Do not subtract the two lag weeks again: that would shift the validation windows.
fold_boundaries = [
    ("2004–2006", 208, 312),
    ("2006–2008", 312, 416),
    ("2008–2010", 416, 520),
]

robustness_models = [
    "Persistence: previous week",
    "Ridge: history + season",
    "Ridge: history + climate",
    "Ridge: history + climate + NDVI",
    "Poisson: history + climate",
]

rolling_results = []
fold_predictions = {}

for fold_name, start, end in fold_boundaries:
    training = model_df.loc[model_df.index < start].copy()
    testing = model_df.loc[
        (model_df.index >= start) & (model_df.index < end)
    ].copy()
    assert len(testing) == 104, "Unexpected validation length"
    assert training.week_start_date.max() < testing.week_start_date.min()

    for model_name in robustness_models:
        spec = candidate_configs[model_name]
        fitted = fit_candidate(spec, training)
        prediction = predict_candidate(fitted, spec, testing)
        score = score_forecast(testing.total_cases, prediction)
        rolling_results.append({
            "fold": fold_name, "model": model_name,
            "train_n": len(training), "valid_n": len(testing),
            "from_date": str(testing.week_start_date.min().date()),
            "to_date": str(testing.week_start_date.max().date()),
            "MAE": score["MAE"], "RMSE": score["RMSE"],
        })
        fold_predictions[(fold_name, model_name)] = (
            testing.total_cases.to_numpy(dtype=float), prediction
        )

rolling_df = pd.DataFrame(rolling_results)
print("Out-of-sample MAE across chronological windows (LOWER = better):")
display(rolling_df.pivot(index="model", columns="fold", values="MAE").round(3))
rolling_df.to_csv(OUTPUT_DIR / "robustness_expanding_windows.csv", index=False)

### 6.7 · Does climate help consistently?

The cleanest test of our environmental research question is to compare **the same Ridge estimator** with and without rainfall, temperature and humidity.

Define **MAE gain = MAE(history + season) − MAE(history + season + climate)**.

- Positive values mean climate **helped** in that validation block.
- Negative values mean climate **hurt** in that block.
- Values close to zero indicate little additional predictive benefit.

We should not automatically equate a positive value in one period with a stable improvement across future years.

In [ ]:
wide = rolling_df.pivot(index="fold", columns="model", values="MAE")
climate_gain = (
    wide["Ridge: history + season"] - wide["Ridge: history + climate"]
).rename("MAE_gain_from_climate")

print("Climate's added value in each historical validation block:")
display(climate_gain.round(3).to_frame())
print("Mean gain across the 3 blocks:", round(climate_gain.mean(), 3), "cases per week")
print("Blocks showing an improvement:", int((climate_gain > 0).sum()), "of", len(climate_gain))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(climate_gain.index, climate_gain.values)
ax.axhline(0, linewidth=1)
ax.set(title="Robustness: climate MAE gain across validation periods",
       xlabel="Validation period", ylabel="History MAE minus climate MAE")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "robustness_climate_gain.png", dpi=170)
plt.show()

### 6.8 · Robustness check 2 — Uncertainty of the small MAE difference

A gain of about **0.05 cases per week** is small. We investigate how sensitive that estimate is to which weeks are represented in the sample.

For the final 104-week internal validation period, compare the weekly **absolute errors** of history-only Ridge and climate Ridge. A *paired moving-block bootstrap* resamples **4-week blocks** of these error differences to retain some short-range dependence.

The resulting interval is **descriptive and approximate**, not a rigorous post-selection statistical significance test. The model was chosen using this validation period, the block length is heuristic, and the data are dependent. The key question is whether a reliable positive gain is evident—not whether a single number is slightly above zero.

In [ ]:
observed, pred_history = fold_predictions[(
    "2008–2010", "Ridge: history + season"
)]
_, pred_climate = fold_predictions[(
    "2008–2010", "Ridge: history + climate"
)]
loss_advantage = np.abs(observed - pred_history) - np.abs(observed - pred_climate)

rng = np.random.default_rng(2026)
B, block_length, n = 3000, 4, len(loss_advantage)
num_blocks = int(np.ceil(n / block_length))
starts = rng.integers(0, n, size=(B, num_blocks, 1))
block_offsets = np.arange(block_length)[None, None, :]
resampled_indices = (starts + block_offsets) % n
resampled_indices = resampled_indices.reshape(B, -1)[:, :n]
bootstrap_gains = loss_advantage[resampled_indices].mean(axis=1)

low, high = np.quantile(bootstrap_gains, [0.025, 0.975])
print("Original internal MAE gain:", round(loss_advantage.mean(), 4))
print("Approx. 95% block-bootstrap interval:", (round(low, 4), round(high, 4)))
print("Fraction of resampled mean gains > 0:", round((bootstrap_gains > 0).mean(), 3))

pd.DataFrame({
    "metric": ["MAE_gain", "bootstrap_2.5_pct", "bootstrap_97.5_pct"],
    "value": [loss_advantage.mean(), low, high],
}).to_csv(OUTPUT_DIR / "robustness_block_bootstrap.csv", index=False)

### 6.9 · Robustness check 3 — Are a few epidemic peaks driving the result?

Large dengue spikes are difficult to predict. To understand their influence, we recalculate the **same predictions'** MAE after excluding the 1, 3, 5, or 10 weeks with the largest *actual* case counts in the **internal validation period**.

This is **a sensitivity analysis of the evaluation metric**, not a new forecasting exercise. We do not refit the model on a reduced dataset. Outbreak weeks remain scientifically important; removing them is only a diagnostic, and we should retain the full-data results as primary.

In [ ]:
peak_rows = []
for k in [0, 1, 3, 5, 10]:
    keep = np.ones(len(observed), dtype=bool)
    if k > 0:
        keep[np.argsort(observed)[-k:]] = False
    h_mae = mean_absolute_error(observed[keep], pred_history[keep])
    c_mae = mean_absolute_error(observed[keep], pred_climate[keep])
    peak_rows.append({
        "excluded_largest_weeks": k,
        "weeks_remaining": int(keep.sum()),
        "history_MAE": h_mae,
        "climate_MAE": c_mae,
        "climate_MAE_gain": h_mae - c_mae,
    })

peak_check = pd.DataFrame(peak_rows)
display(peak_check.round(3))
peak_check.to_csv(OUTPUT_DIR / "robustness_extreme_week_sensitivity.csv", index=False)

### 6.10 · Interpretation of the robustness evidence

Write your final methodological discussion **after running these cells**. Focus on three things:

1. **Temporal stability:** Did climate improve MAE in all three chronological periods, or only some?
2. **Uncertainty:** Does the approximate interval for the small MAE gain remain above zero?
3. **Sensitivity to extreme weeks:** Does excluding a handful of extreme dengue weeks substantially change the comparison?

**What a cautious conclusion looks like:** The climate-enhanced Ridge model shows a small improvement in the originally selected internal period and in the later-period check, but the effect varies across historical blocks. We therefore regard its additional predictive contribution as **modest and not consistently established**, not as proof that weather strongly determines outbreaks.

Do not modify or reselect the chosen model based on these diagnostics or on the CDC observations. The next sections report the previously chosen model on the later period for transparency.

## 7 · External-period data: CDC observed case counts

**Only now** do we open the 2010–2013 surveillance labels. We must first check that the independent public case series measures the same outcome as DengAI's earlier `total_cases`.

The CDC-derived CSV has:
- `week_start_date` — date of the observation week.
- `total_cases_cdc` — reported dengue cases for that week.

The later `dengue_features_test.csv` contains environmental predictors but **no case counts**. We match its 156 Iquitos rows to the CDC outcomes by date.

**The 520 historical overlapping weeks must match exactly**, otherwise we should not use the CDC series as comparable labels without investigating definitions and dates.

In [ ]:
cdc = pd.read_csv(DATA_DIR / "Iquitos_CDC_Observed_Cases_2000_2013.csv")
cdc["week_start_date"] = pd.to_datetime(cdc["week_start_date"])
assert cdc["week_start_date"].is_unique
assert len(cdc) == 676

check = iq[["week_start_date", "total_cases"]].merge(
    cdc, on="week_start_date", how="left", validate="one_to_one"
)
assert len(check) == 520 and check["total_cases_cdc"].notna().all()
assert (check["total_cases"] == check["total_cases_cdc"]).all(), "CDC and DengAI labels differ"
print("All", len(check), "labeled DengAI weeks MATCH the CDC public series.")

future = features_test.loc[features_test["city"].eq("iq")].copy()
future["week_start_date"] = pd.to_datetime(future["week_start_date"])
future = future.sort_values("week_start_date").reset_index(drop=True)
assert len(future) == 156
assert future["week_start_date"].is_unique
assert not set(future["week_start_date"]).intersection(iq["week_start_date"])
future = future.merge(cdc, on="week_start_date", how="left", validate="one_to_one")
assert len(future) == 156 and future["total_cases_cdc"].notna().all()
print("Later labeled weeks:", len(future))
print("Period:", future.week_start_date.min().date(), "to", future.week_start_date.max().date())
display(future[["week_start_date", "total_cases_cdc"]].head())

## 8 · External-period evaluation — no new model selection

### 8.1 Retrain the frozen specification on all 520 labeled DengAI weeks

After selecting the best model **internally**, we are allowed to retrain the **same specification** using all original history (2000–2010). The external case counts are **not included in fitting**.

### Important: what does “external one-week-ahead forecast” mean here?

We construct lagged inputs across the combined historical timeline. For a target week in 2011, the forecast is allowed to use **observed dengue cases from the preceding week** (published later by CDC) under a *retrospective rolling* information assumption. It must not use the target week's count as a predictor.

Therefore this is **an independent later-period evaluation of a rolling one-week-ahead procedure**, NOT a single 156-week-ahead forecast made at the start of July 2010.

A real-time system may not have such prompt access to recent dengue counts because of reporting delays.

In [ ]:
future_labeled = future.rename(columns={"total_cases_cdc": "total_cases"})
whole_timeline = pd.concat([iq, future_labeled], ignore_index=True)
whole_timeline = whole_timeline.sort_values("week_start_date").reset_index(drop=True)
assert len(whole_timeline) == 676

all_features = make_features(whole_timeline)
external_rows = all_features.iloc[520:].copy()
assert len(external_rows) == 156 and external_rows["total_cases"].notna().all()

# The model is retrained ONLY on earlier DengAI weeks, using the frozen choice.
final_selected_model = fit_candidate(selected_spec, model_df)
selected_external_forecast = predict_candidate(final_selected_model, selected_spec, external_rows)

# Two PREDECLARED reference forecasts for scientific interpretation (not selection):
history_only_spec = candidate_configs["Ridge: history + season"]
final_history_model = fit_candidate(history_only_spec, model_df)
history_external_forecast = predict_candidate(final_history_model, history_only_spec, external_rows)
persistence_external_forecast = external_rows["cases_lag1"].to_numpy(dtype=float)

print("Frozen selected specification:", selected_model_name)
print("Model-fitting rows:", len(model_df), "| Later evaluation weeks:", len(external_rows))
# Save later-period inputs with retrospective observed lag counts.
external_rows.to_csv(PROCESSED_DIR / "iquitos_external_features.csv", index=False)


### 8.2 Score predictions against the later observed cases

We use exactly the same metrics as for internal validation.

- **MAE:** average absolute prediction error, measured in *cases/week*.
- **RMSE:** square root of average squared prediction error; more sensitive to large misses during outbreaks.

The model with the lowest MAE was already selected. External comparison tells us how that choice **generalizes to later years**; we do not replace the selected model based on this table.

In [ ]:
external_observed = external_rows["total_cases"].to_numpy(dtype=float)
external_results = pd.DataFrame([
    {"Model": selected_model_name, **score_forecast(external_observed, selected_external_forecast)},
    {"Model": "History-only Ridge reference", **score_forecast(external_observed, history_external_forecast)},
    {"Model": "Persistence reference", **score_forecast(external_observed, persistence_external_forecast)},
])
print("EXTERNAL PERIOD — 2010–2013 (selected model was frozen above)")
display(external_results.round(3))
external_results.to_csv(OUTPUT_DIR / "external_model_comparison.csv", index=False)

### 8.3 Visual comparison of forecasts and the observed outbreak weeks

Do our forecasts follow ordinary weeks? What happens during sudden epidemic peaks? A model may have good overall MAE and still underestimate rare large outbreaks.

In [ ]:
external_predictions = pd.DataFrame({
    "week_start_date": external_rows["week_start_date"].to_numpy(),
    "observed_cases_cdc": external_observed.astype(int),
    "selected_model_forecast": selected_external_forecast,
    "history_only_forecast": history_external_forecast,
    "persistence_forecast": persistence_external_forecast,
})
external_predictions["absolute_error_selected"] = np.abs(
    external_predictions["observed_cases_cdc"] - external_predictions["selected_model_forecast"]
)
external_predictions.to_csv(OUTPUT_DIR / "external_predictions.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 4.4))
ax.plot(external_predictions.week_start_date, external_predictions.observed_cases_cdc, label="Observed CDC cases", linewidth=2)
ax.plot(external_predictions.week_start_date, external_predictions.selected_model_forecast, label="Frozen selected forecast", alpha=0.85)
ax.set(title="External-period evaluation (2010–2013)", xlabel="Date", ylabel="Dengue cases per week")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "external_validation.png", dpi=170)
plt.show()

print("Largest observed weeks in the external period:")
display(external_predictions.nlargest(5, "observed_cases_cdc")[[
    "week_start_date", "observed_cases_cdc", "selected_model_forecast", "absolute_error_selected"
]].round(2))

## 9 · Final research interpretation and reproducibility

### 9.1 What can we claim?

The comparison asks whether **previous weather and vegetation measurements improve the ability to forecast weekly dengue cases** beyond recent case counts and calendar seasonality. This is a **supervised time-series prediction study**, not causal inference.

**What we cannot claim:**
- That rainfall, temperature or NDVI *caused* an outbreak.
- That NDVI is a direct measurement of environmental degradation.
- That the same forecasting performance will hold in every future decade or city.
- That a rolling one-week-ahead evaluation equals a multi-year forecast with no new surveillance reports.

The public CDC data are **another publication of overlapping Iquitos surveillance records**, so we call this **later-period (out-of-time) evaluation**, not independent geographical validation.

### 9.2 What to include in the report

**Introduction:** Topic 3, environmental epidemiology and importance of anticipating dengue outbreaks.  
**Data:** DengAI + CDC; Iquitos; calendar range, measured variables and missingness.  
**EDA:** dengue and climate/vegetation graphs with cautious observations.  
**Methods:** one-week-ahead horizon, lagged variables, two-stage chronological validation, candidate regression models, MAE and RMSE.  
**Results:** internal candidate comparison; selected model; external later-period scores and error plots.  
**Discussion:** environmental predictive relevance, small improvements if observed, failures around outbreaks, reporting delays, limitations.  
**Conclusion:** answer only the research question supported by the evaluations.

### Robustness addendum
The dedicated robustness section (6.6–6.10) assesses whether the very small climate MAE gain is stable across periods. These checks are exploratory and do not change the already-selected model.

In [ ]:
summary_rows = [
    {"Stage": "Internal (2008–2010)", "Model": selected_model_name,
     **score_forecast(val_df["total_cases"], internal_predictions[selected_model_name])},
]
for _, row in external_results.iterrows():
    summary_rows.append({"Stage": "External later-period (2010–2013)",
                         "Model": row["Model"], "MAE": row["MAE"], "RMSE": row["RMSE"]})
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / "research_results_summary.csv", index=False)
print("RESEARCH SUMMARY")
display(summary.round(3))
print("\nSaved figures and tables to:", OUTPUT_DIR.resolve())
print("Remember: these are retrospective rolling one-week-ahead predictions.")